# Persistência do modelo financeiro de demonstração
Reutiliza a tabela Gold do teste Medallion anterior, treina o pipeline Spark ML e salva em Object Storage.
O pipeline salvo inclui StringIndexer, VectorAssembler, StandardScaler e RandomForestClassifier.
Recarrega com PipelineModel.load e exige igualdade de predições e probabilidades em 100 exemplos.
Os rótulos são sintéticos/aleatórios. O modelo não é adequado para decisões reais de fraude e não é publicado como endpoint.

In [ ]:
import json, math, re
from uuid import uuid4
import pyspark.sql.functions as F
if 'spark' not in globals() or 'oidlUtils' not in globals():
    raise RuntimeError('Execute no runtime AIDP com compute anexado.')
def parameter(name, default):
    value = oidlUtils.parameters.getParameter(name, str(default))
    return str(default) if value is None else str(value).strip()
source_table = parameter('SOURCE_TABLE', 'default.finance_gold_dc473dcc1167.fraud_detection_model_gold')
if not re.fullmatch(r'[A-Za-z_][A-Za-z0-9_]*\.[A-Za-z_][A-Za-z0-9_]*\.[A-Za-z_][A-Za-z0-9_]*', source_table):
    raise ValueError('SOURCE_TABLE deve ser catalog.schema.table.')
seed = int(parameter('SEED', 42))
num_trees, max_depth = 100, 10
run_id = uuid4().hex[:12]
model_path = f'oci://aidp_test@idzvuvikb5ym/aidp_codex_tests/models/finance_rf/{run_id}/spark_pipeline'
ml_features = spark.table(source_table)
result = {'run_id': run_id, 'source_table': source_table, 'artifact_path': model_path, 'spark_version': spark.version, 'synthetic_labels_only': True, 'serialization': 'spark_ml_pipeline'}
print('FINANCE_MODEL_STAGE=source_loaded ' + json.dumps(result))

In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler, StandardScaler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator

ml_input = ml_features.withColumn('amount_ml', F.col('amount').cast('double'))
train_data, test_data = ml_input.randomSplit([0.8, 0.2], seed=seed)
train_data, test_data = train_data.cache(), test_data.cache()
train_count, test_count = train_data.count(), test_data.count()
if train_count == 0 or test_count == 0 or train_data.select('is_fraud').distinct().count() != 2 or test_data.select('is_fraud').distinct().count() != 2:
    raise ValueError('Treino e teste devem ser não vazios e conter ambas as classes.')
indexers = [StringIndexer(inputCol=c, outputCol=c + '_index', handleInvalid='keep') for c in ('transaction_type', 'merchant_category', 'location')]
feature_names = ['amount_ml','month','day_of_week','hour','transaction_type_index','merchant_category_index','location_index']
assembler = VectorAssembler(inputCols=feature_names, outputCol='features')
scaler = StandardScaler(inputCol='features', outputCol='scaled_features')
rf = RandomForestClassifier(labelCol='is_fraud', featuresCol='scaled_features', numTrees=num_trees, maxDepth=max_depth, seed=seed)
pipeline = Pipeline(stages=indexers + [assembler, scaler, rf])
print(f'FINANCE_STAGE=training_started train={train_count} test={test_count} trees={num_trees}')
model = pipeline.fit(train_data)
predictions = model.transform(test_data).cache()
auc = BinaryClassificationEvaluator(labelCol='is_fraud', rawPredictionCol='rawPrediction', metricName='areaUnderROC').evaluate(predictions)
assert math.isfinite(auc) and 0 <= auc <= 1
assert predictions.count() == test_count
confusion = {(int(r.is_fraud), int(r.prediction)): r['count'] for r in predictions.groupBy('is_fraud','prediction').count().collect()}
tp, tn = confusion.get((1,1),0), confusion.get((0,0),0)
fp, fn = confusion.get((0,1),0), confusion.get((1,0),0)
result.update(train_rows=train_count, test_rows=test_count, rf_num_trees=num_trees, rf_max_depth=max_depth, auc=float(auc),
    accuracy=(tp+tn)/test_count, precision=tp/(tp+fp) if tp+fp else 0.0, recall=tp/(tp+fn) if tp+fn else 0.0,
    confusion_matrix={f'actual_{a}_pred_{p}':count for (a,p),count in confusion.items()},
    feature_importance={name:float(value) for name,value in zip(feature_names, model.stages[-1].featureImportances)})
print('FINANCE_STAGE=training_complete ' + json.dumps(result, default=str))

In [ ]:
from pyspark.ml import PipelineModel
# Caminho exclusivo: não sobrescreve nenhum modelo anterior.
model.write().save(model_path)
print('FINANCE_MODEL_STAGE=saved ' + model_path)
loaded_model = PipelineModel.load(model_path)
key_columns = ['account_id','transaction_date','amount','transaction_type','merchant_category','location','month','day_of_week','hour']
probe = test_data.orderBy(*key_columns).limit(100).cache()
probe_count = probe.count()
if probe_count == 0:
    raise AssertionError('A amostra de validação do modelo salvo está vazia.')
def scoring_rows(pipeline_model):
    rows = pipeline_model.transform(probe).select(*key_columns,'prediction','probability').collect()
    return sorted([tuple(str(row[c]) for c in key_columns) + (float(row.prediction), tuple(float(p) for p in row.probability)) for row in rows])
if scoring_rows(model) != scoring_rows(loaded_model):
    raise AssertionError('Predições ou probabilidades diferem após salvar/carregar.')
result.update(status='SUCCESS', persisted=True, reload_verified=True, validation_rows=probe_count)
spark.createDataFrame([(json.dumps(result,sort_keys=True),)], ['value']).coalesce(1).write.mode('errorifexists').text(model_path.rsplit('/',1)[0] + '/validation')
print('AIDP_FINANCE_MODEL_PERSISTENCE_RESULT=' + json.dumps(result, sort_keys=True))
probe.unpersist()
predictions.unpersist()
train_data.unpersist()
test_data.unpersist()